1. VERIFICAÇÃO DO BANCO DE DADOS

In [1]:
file.exists("/content/exportacao_cargas.csv.gz")

[1] TRUE

2. IMPORTAÇÃO DO BANCO DE DADOS

In [2]:
dados <- read.csv(
  "/content/exportacao_cargas.csv.gz",
  stringsAsFactors = FALSE,
  check.names = FALSE
)

3. VERIFICANDO SE O BANCO DE DADOS FOI IMPORTADO (LINHAS E COLUNAS)

In [3]:
dim(dados)

[1] 1097014      16

4. CRIANDO VARIÁVEL Y COM CARGA CONTERIZA (NATUREZA_CARGA)

In [4]:
dados$y <- as.integer(dados$NATUREZA_CARGA == "CARGA CONTEINERIZADA")
table(dados$y)


     0      1 
118694 978320 

5. TRANSFORMANDO TOTAL_TONELADAS EM NÚMERO (TIRANDO A VÍRGUAL E COLOCANDO PONTO, (QUALITATIVA CONTÍNUA))

In [5]:
dados$TOTAL_TONELADAS <- as.numeric(
  gsub(",", ".", dados$TOTAL_TONELADAS)
)

6. CRIANDO X1 (TOTAL_TONELADAS) E X2 (TOTAL_UNID)

In [6]:
dados$x1 <- dados$TOTAL_TONELADAS
dados$x2 <- dados$TOTAL_UNID

7. REDUZINDO PARA UMA AMOSTRA

In [7]:
set.seed(123)
dados <- dados[sample(nrow(dados), 5000), ]

8. AJUSTANDO A LOGÍSTICA E LENDO RAZÕES DE CHANCE

In [8]:
m <- glm(y ~ x1 + x2, family = binomial, data = dados)
summary(m);  exp(coef(m))

Warning message:
“glm.fit: fitted probabilities numerically 0 or 1 occurred”



Call:
glm(formula = y ~ x1 + x2, family = binomial, data = dados)

Coefficients:
              Estimate Std. Error z value Pr(>|z|)    
(Intercept)  2.886e+00  6.517e-02   44.28  < 2e-16 ***
x1          -2.081e-04  1.189e-05  -17.50  < 2e-16 ***
x2          -7.787e-04  1.599e-04   -4.87 1.12e-06 ***
---
Signif. codes:  0 ‘***’ 0.001 ‘**’ 0.01 ‘*’ 0.05 ‘.’ 0.1 ‘ ’ 1

(Dispersion parameter for binomial family taken to be 1)

    Null deviance: 3543.9  on 4999  degrees of freedom
Residual deviance: 2467.0  on 4997  degrees of freedom
AIC: 2473

Number of Fisher Scoring iterations: 10


(Intercept)          x1          x2 
 17.9165802   0.9997920   0.9992216

9. PREVENDO PROBABILIDADE E CLASSE (LIMIAR 0.5)

In [9]:
p    <- predict(m, type = "response")
yhat <- as.integer(p > 0.5)
table(real = dados$y, previsto = yhat)

    previsto
real    0    1
   0  214  355
   1   48 4383

10. REPETINDO COM LIMIAR 0.3 E 0.7

In [10]:
yhat03 <- as.integer(p > 0.3)
table(real = dados$y, previsto = yhat03)

yhat07 <- as.integer(p > 0.7)
table(real = dados$y, previsto = yhat07)

    previsto
real    0    1
   0  187  382
   1   21 4410

    previsto
real    0    1
   0  238  331
   1  105 4326

11. AUC

In [11]:
mean(outer(p[dados$y == 1], p[dados$y == 0], ">"))

[1] 0.8557197